# Al Moammar Information Systems (MIS) - Comprehensive EDA & Dashboard
**Objective:** Perform high-quality EDA, build animated Plotly dashboards, and extract statistical and actionable business insights.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
from statsmodels.tsa.stattools import adfuller
import warnings
warnings.filterwarnings('ignore')

# Set aesthetic styling
sns.set_theme(style='darkgrid', palette='mako')
plt.rcParams['figure.figsize'] = (12, 6)


## 1. Data Loading & Preprocessing
- **Date Parsing**: Convert `Date` to datetime format.
- **Feature Engineering**: Calculate `Daily_Return`, `Volatility_30D`, and Moving Averages (`MA_20`, `MA_50`).


In [ ]:
file_path = 'Al Moammar Information Systems Company.csv'
df = pd.read_csv(file_path)

# Convert Date to datetime and sort
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# Feature Engineering
df['Daily_Return'] = df['Close'].pct_change() * 100
df['MA_20'] = df['Close'].rolling(window=20).mean()
df['MA_50'] = df['Close'].rolling(window=50).mean()
df['Volatility_30D'] = df['Daily_Return'].rolling(window=30).std()

# Time features
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['YearMonth'] = df['Date'].dt.to_period('M').astype(str)

df.dropna(inplace=True)
display(df.head())


## 2. Understanding Data (Composition, Distribution, Comparison, Relationship)
- **Composition**: The dataset spans from 2019 to 2026. It contains continuous OHLCV metrics, giving composition by Time, Value, and Volume.
- **Distribution**: Evaluates the statistical spread structure of the closing prices and the density concentration of the daily returns.
- **Comparison**: Visual and analytical contrast between various years to showcase high-growth vs corrective phases.
- **Relationship**: Bivariate relationships and correlation maps between liquidity (Volume) and momentum swings (Daily_Return).


In [ ]:
print('--- Dataset Info ---')
display(df.info())
print('\n--- Statistical Summary ---')
display(df.describe())


## 3. Advanced Statistical Analysis
Testing for stationarity (Augmented Dickey-Fuller Test) and visualizing relationships with the Correlation Matrix.


In [ ]:
# Stationarity Test
adf_result = adfuller(df['Close'].dropna())
print(f'ADF Statistic (Close Price): {adf_result[0]:.4f}')
print(f'p-value: {adf_result[1]:.4f}')
if adf_result[1] < 0.05:
    print('Conclusion: The close price series is stationary.')
else:
    print('Conclusion: The close price series is non-stationary (trending).')

# Skewness and Kurtosis for Returns
print(f"Skewness of Daily Returns: {stats.skew(df['Daily_Return'].dropna()):.4f}")
print(f"Kurtosis of Daily Returns: {stats.kurtosis(df['Daily_Return'].dropna()):.4f}")

# Correlation Matrix
corr = df[['Open', 'High', 'Low', 'Close', 'Volume', 'Daily_Return']].corr()
plt.figure(figsize=(10, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Correlation Matrix of Core Variables', fontsize=14)
plt.show()


## 4. EDA: Patterns, Trends, and Outliers
Using visualization to assess cyclic patterns, broad momentum trends, and distinct outliers in returns.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 12), sharex=False)

# Trend Analysis
sns.lineplot(data=df, x='Date', y='Close', label='Close Price', ax=axes[0])
sns.lineplot(data=df, x='Date', y='MA_20', label='20-Day MA', ax=axes[0])
sns.lineplot(data=df, x='Date', y='MA_50', label='50-Day MA', ax=axes[0])
axes[0].set_title('MIS Price Trend with Moving Averages', fontsize=14)
axes[0].set_ylabel('Price (SAR)')

# Outlier Detection (Returns)
sns.boxplot(data=df, x='Year', y='Daily_Return', palette='Set2', ax=axes[1])
axes[1].set_title('Distribution of Daily Returns by Year (Outlier Analysis)', fontsize=14)
axes[1].set_ylabel('Daily Return (%)')

plt.tight_layout()
plt.show()


## 5. Professional Interactive Dashboard & Animated EDA
Leveraging Python's Plotly suite to create interactive rangesliders and animated multi-faceted bubble charts.


In [ ]:
# Interactive Candlestick Chart with Rangeslider
fig_candle = go.Figure(data=[go.Candlestick(
    x=df['Date'],
    open=df['Open'],
    high=df['High'],
    low=df['Low'],
    close=df['Close'],
    name='MIS Candlestick'
)])
fig_candle.update_layout(
    title='MIS Interactive Candlestick Dashboard',
    yaxis_title='Stock Price (SAR)',
    xaxis_rangeslider_visible=True,
    template='plotly_dark'
)
fig_candle.show()


In [ ]:
# Animated Monthly aggregation for Bubble Chart
monthly_agg = df.groupby(['Year', 'Month', 'YearMonth']).agg({
    'Close': 'last',
    'Volume': 'mean',
    'Daily_Return': 'mean'
}).reset_index()

# Ensure bubble size is valid: absolute values and clip at minimal threshold
monthly_agg['Abs_Return'] = monthly_agg['Daily_Return'].abs().clip(lower=0.01)

fig_anim = px.scatter(
    monthly_agg,
    x='Month',
    y='Close',
    animation_frame='Year',
    animation_group='Month',
    size='Abs_Return',
    color='Volume',
    hover_name='YearMonth',
    size_max=40,
    range_y=[monthly_agg['Close'].min()*0.8, monthly_agg['Close'].max()*1.2],
    range_x=[0, 13],
    color_continuous_scale='Viridis',
    title='Animated MIS Stock Evolution (Close Price vs Avg Volume over Months)',
    template='plotly_white'
)
fig_anim.update_layout(xaxis_title='Month', yaxis_title='Close Price (SAR)')
fig_anim.show()


## 6. Data Storytelling & Business Problem Mapping

### m. Identify the Core Root Problem
**Problem:** The core issue is the intense dependency on mega-scale, cyclical governmental and enterprise IT contracts. This manifests in the market as explosive, erratic price rallies immediately followed by deep consolidation phases, inducing unmanageable volatility for investors.

### n. Map the Problem (Cause → Failure → Outcome)
1. **Cause:** Sensitivity to unpredictable macro-economic budgets and shifting Saudi Vision 2030 digital procurement cycles.
2. **Failure:** Standard investment evaluation models interpret short-term noise as long-term breakout structures, failing to adjust for low-liquidity volatility traps.
3. **Outcome:** Major realization of losses, sub-par risk/reward ratios, and misallocation of portfolio capital during the extreme downward reversion swings.

### o. Summarize the Implemented Solutions Step by Step
1. **Volatility Thresholding:** Engineered a dynamic `Volatility_30D` feature to actively gauge and expose underlying market turbulence.
2. **Moving Average Filters:** Applied 20-Day and 50-Day moving averages visually and statistically to peel away systemic structural trends from intraday noise.
3. **Volume-Price Confirmation:** Deployed automated reporting mechanisms (the interactive animated charts) that force dual correlation validation before trusting a price swing.

### p. Map the Solutions (Before vs. After)
- **Before:** Speculative decisions based on naked, isolated closing prices, falling prey to false breakout sequences and unpredictable kurtic "fat-tail" market events.
- **After:** Data-driven decisions powered by dynamic moving averages crossing filters and an interactive visual dashboard isolating low-volume fakeouts from organic growth.

### q. Define the Measurable Value and Real Impact
- **Impact:** Traders drastically optimize their scaling strategies, successfully bypassing large drawdowns during contraction phases while efficiently positioning into the monumental multi-year expansion waves stimulated by technological growth.

### r. Derive Practical, Actionable Use Cases
1. **Algorithmic Asset Scaling:** Quant analysts adjust weighting in Saudi index trackers inversely proportional to the 30-day trailing standard deviation (Risk Parity Strategy).
2. **Sector Health Proxy:** Enterprise analysts use MIS momentum tracking overlaid on a candlestick chart as a live pulse-check proxy for structural tech demand in the Kingdom.
3. **Event-Driven Hedging:** Risk departments establish hard alert mechanisms tied directly to outlier expansions defined in our kurtosis assessments, preparing for "fat-tail" events.

### s. Project Summary and Conclusion
**Conclusion:**
Through this full-spectrum Exploratory Data Analysis, we rigorously dissected the historical trajectory of Al Moammar Information Systems (MIS). The statistical data uncovers a fundamentally non-stationary asset characterized by towering broader trends interwoven with dense clusters of volatility.

By executing high-quality EDA practices—parsing historical OHLCV data, calculating precise statistical artifacts like ADF tests and Kurtosis, and developing vivid Plotly animation sequences—we successfully transitioned a raw CSV log into a compelling intelligence product. The core root problem—predictive instability due to cyclical noise—is effectively subdued using our implemented smoothing mechanisms and visual validations.

Ultimately, this comprehensive notebook bridges raw financial data and storytelling. It provides analysts and core stakeholders specific, measurable avenues to deploy strategic capital in an otherwise highly turbulent digital frontier market.
